In [22]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error,r2_score

from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

##### Load data and select feature

In [23]:
df = pd.read_csv('boston_data.csv')

In [24]:
df.head()

,ID,indus,chas,nox,rm,age,dis,rad,tax,ptratio,lstat,medv
0,1,2.31,0,0.538,6.575,65.2,4.0900,1,296,15.3,4.98,24.0
1,2,7.07,0,0.469,6.421,78.9,4.9671,2,242,17.8,9.14,21.6
2,4,2.18,0,0.458,6.998,45.8,6.0622,3,222,18.7,2.94,33.4
3,5,2.18,0,0.458,7.147,54.2,6.0622,3,222,18.7,5.33,36.2
4,7,7.87,0,0.524,6.012,66.6,5.5605,5,311,15.2,12.43,22.9


##### Split the input and output 

In [25]:
X = df.iloc[:,1:-1]
Y = df['medv']

In [26]:
X.head()

,indus,chas,nox,rm,age,dis,rad,tax,ptratio,lstat
0,2.31,0,0.538,6.575,65.2,4.0900,1,296,15.3,4.98
1,7.07,0,0.469,6.421,78.9,4.9671,2,242,17.8,9.14
2,2.18,0,0.458,6.998,45.8,6.0622,3,222,18.7,2.94
3,2.18,0,0.458,7.147,54.2,6.0622,3,222,18.7,5.33
4,7.87,0,0.524,6.012,66.6,5.5605,5,311,15.2,12.43


In [27]:
Y.head()

0    24.0
1    21.6
2    33.4
3    36.2
4    22.9
Name: medv, dtype: float64

##### Train test split

In [28]:
X_train,X_test,Y_train,Y_test = train_test_split(X,Y,test_size=0.2,random_state = 42)

In [29]:
X_train.shape,X_test.shape,Y_train.shape,Y_test.shape

((266, 10), (67, 10), (266,), (67,))

In [30]:
X_train.head()

,indus,chas,nox,rm,age,dis,rad,tax,ptratio,lstat
224,5.19,0,0.515,5.985,45.4,4.8122,5,224,20.2,9.74
78,10.01,0,0.547,6.021,82.6,2.7474,6,432,17.8,10.30
295,18.10,0,0.713,6.525,86.5,2.4358,24,666,20.2,18.13
17,8.14,0,0.538,6.047,88.8,4.4534,4,307,21.0,17.28
24,6.91,0,0.448,6.169,6.6,5.7209,3,233,17.9,5.81


##### preprocessor with ColumnTransformer

In [31]:
df.columns

Index(['ID', 'indus', 'chas', 'nox', 'rm', 'age', 'dis', 'rad', 'tax',
       'ptratio', 'lstat', 'medv'],
      dtype='object')

In [32]:
bin_features = ['chas']
num_features = X.drop(columns=['chas']).columns
print(bin_features)
print(num_features)

['chas']
Index(['indus', 'nox', 'rm', 'age', 'dis', 'rad', 'tax', 'ptratio', 'lstat'], dtype='object')


In [33]:
def build_preprocessor():
    return ColumnTransformer(transformers=[
    ('num',StandardScaler(),num_features),
    ("bin", 'passthrough', bin_features)
    ])

def build_pipeline(model):
    return Pipeline(steps=[
        ('preprocessor',build_preprocessor()),
        ('model',model),
    ])

In [34]:
def evaluate(pipe,X_te,y_te):
    pred = pipe.predict(X_te)
    return{
    'r2_score' : round(r2_score(y_te,pred)*100,2)
    }

In [35]:
baseline_model = {
    "Linear Regression":LinearRegression(),
    "KNN":KNeighborsRegressor(),
    "Decision Tree": DecisionTreeRegressor(random_state=42),
    "Random forest":RandomForestRegressor(random_state=42)
}

In [36]:
baseline_results = {}
for name,model in baseline_model.items():
    pipe = build_pipeline(model)
    pipe.fit(X_train,Y_train)
    scores = evaluate(pipe,X_test,Y_test)
    baseline_results[name] = scores

In [37]:
baseline_results

{'Linear Regression': {'r2_score': 72.92},
 'KNN': {'r2_score': 77.88},
 'Decision Tree': {'r2_score': 67.51},
 'Random forest': {'r2_score': 89.88}}

##### test model on random data

In [19]:
X_test.iloc[12]

indus       18.100
chas         0.000
nox          0.583
rm           5.871
age         41.900
dis          3.724
rad         24.000
tax        666.000
ptratio     20.200
lstat       13.340
Name: 319, dtype: float64

##### Saving model model

In [20]:
import joblib

In [21]:
joblib.dump(baseline_model,'BL_model.pkl')

['BL_model.pkl']